# TransitPulse MBTA — demo full-stack trên Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nqMinh-GR/de_btl/blob/main/notebooks/TransitPulse_MBTA_Colab_Demo.ipynb)

Open in Colab: `https://colab.research.google.com/github/nqMinh-GR/de_btl/blob/main/notebooks/TransitPulse_MBTA_Colab_Demo.ipynb`

## Bài toán

Theo dõi xe buýt MBTA gần thời gian thực và cảnh báo **độ trễ dự kiến** của các chuyến trên một phạm vi nhỏ (tối đa 5 tuyến bus). Notebook là *entrypoint* của demo: nó clone đúng commit, dựng runtime native trong VM Colab, chạy pipeline thật và hiển thị kết quả qua API. Notebook **không** chứa logic nghiệp vụ hay tính KPI.

## Kiến trúc (native trong VM Colab, không Docker)

| Tầng | Thành phần |
|---|---|
| Nguồn | GTFS static + GTFS-Realtime (VehiclePositions, TripUpdates, Alerts) — live MBTA, replay recording, hoặc synthetic có seed |
| Ingestion | collector (live) hoặc replay/synthetic publisher → raw archive + outbox → Kafka 4.x (KRaft) |
| Xử lý | Spark Structured Streaming 3.5 (Python 3.11 trong `.venv`): parse, dedup, freshness, KPI, chất lượng |
| Lưu trữ | Cassandra 5: generation READY + con trỏ (publish nguyên tử), lịch sử xe, chất lượng |
| Phục vụ | FastAPI (`127.0.0.1:8000/api/v1`) → UI notebook 5 tab (Colab) và dashboard Streamlit + PyDeck (chỉ đọc, có thể mở qua link công khai) |

Mọi dịch vụ chạy nền do *supervisor* quản lý (PID registry, readiness có timeout); cell trong notebook chỉ gọi lệnh ngắn, có giới hạn, nên **Run All** không bị treo. UI trong notebook gọi API từ kernel Python qua HTTP; trình duyệt không cần truy cập `localhost:8000`.

## Ba mode dữ liệu (`DEMO_MODE`)

* **synthetic** (mặc định): recording sinh tất định từ seed, phát qua publisher → Kafka → Spark → Cassandra → API thật. Banner tím **SYNTHETIC — Dữ liệu kiểm thử — không phải xe MBTA thật**. 5 tuyến synthetic **không** phải 5 tuyến MBTA đã xác minh. Đây là demo tích hợp, không phải bằng chứng đã thu dữ liệu thật.
* **replay**: phát lại recording đã ghi (manifest + raw + static ZIP đúng version, checksum được kiểm) với đồng hồ nguồn và tốc độ `REPLAY_SPEED`; banner hiển thị đồng hồ lịch sử và thời điểm phục vụ (wall clock). Replay một recording synthetic vẫn mang nhãn SYNTHETIC. Thiếu manifest → lỗi có hướng dẫn, không tự sinh dữ liệu giả.
* **live**: gọi MBTA thật cho 1–5 bus route đã kiểm tra (`ROUTE_IDS`), poll 30 s, đồng hồ wall. Nguồn lỗi → báo source failure/stale; notebook **không bao giờ** tự chuyển sang synthetic dưới nhãn LIVE.

## Ý nghĩa các chỉ số (do Spark tính, API trả về, UI chỉ hiển thị)

* **Xe có vị trí mới (fresh)**: xe có quan sát với tuổi ≤ 120 s tại đồng hồ ứng dụng — *không* phải toàn bộ đội xe MBTA đang chạy. Ngoài ra có *stale*, *không rõ tuổi* (thiếu timestamp) và *invalid* (tọa độ/thời gian sai, không vẽ lên bản đồ).
* **Độ trễ dự kiến ≥ 5 phút: X trên Y chuyến đủ điều kiện**: mỗi chuyến (trip instance) góp một phiếu tại stop mục tiêu `first_future_resolved_stop_v1`; chuyến chỉ theo lịch, degraded, invalid, bị hủy hoặc không phân giải được lịch bị loại và đếm theo lý do. Mẫu số bằng 0 → hiển thị **“Chưa có mẫu đủ điều kiện”**, không bao giờ hiển thị 0 %.
* Giá trị thiếu hiển thị “—”, không bao giờ thay bằng 0. Lỗi API (503, hết hạn, rỗng, không kết nối) có trạng thái riêng.

## Mục tiêu ≠ kết quả đo

Các mốc trong tài liệu (poll 30 s, stale 120/90/300 s, độ trễ hiển thị p95, throughput…) là **policy/mục tiêu của dự án**, chưa phải số đã đo. Kết quả thật chỉ có trong report sinh ra khi chạy (unit tests, golden smoke, benchmark, fault scenarios); bước nào chưa chạy được ghi **NOT_RUN** — không có bảng PASS mẫu.

Test, golden smoke, benchmark hay export thất bại **không** làm dừng Run All (để session, UI và link vẫn dùng được), nhưng được ghi **FAIL** trong `reports/notebook/notebook_steps.json`, hiện khung đỏ ngay tại cell đó và tóm tắt đỏ ở cell 14, 17, 18. Đặt `STRICT_RUN_ALL = True` để cell 18 báo lỗi khi còn bước FAIL.

## Link công khai tới dashboard (Cloudflare quick tunnel)

* Với `ENABLE_CLOUDFLARE_TUNNEL = True` (mặc định), cell 14 chạy `demo.py expose start` để mở dashboard Streamlit **chỉ đọc** qua một URL dạng `https://….trycloudflare.com`. Không cần tài khoản hay token Cloudflare. `TUNNEL_TARGET` chọn `dashboard` (mặc định), `api` hoặc `both`. Với `False`, bootstrap chạy kèm `--no-tunnel`, chỉ bỏ qua cloudflared (dashboard extra vẫn luôn được cài), và cell 14 đóng mọi tunnel còn sót từ lần chạy trước.
* **Bảo mật:** link là **công khai** và không có đăng nhập — **bất kỳ ai có link đều xem được** dashboard (và API chỉ đọc nếu `TUNNEL_TARGET` là `api`/`both`) chừng nào session còn chạy. Chỉ chia sẻ khi chấp nhận điều đó; không dùng cho dữ liệu nhạy cảm. Notebook không ghi link vào notebook_steps.json.
* Đóng link bằng nút **Tắt tunnel** (`demo.py expose stop --target both`, đóng cả dashboard và API) hoặc **Dừng tất cả** (`demo.py stop --all`) ở cell 18. “Dừng session” (`demo.py stop`) vẫn giữ API, dashboard và tunnel.
* DNS của link mới có thể mất tới khoảng 1 phút (đó là cảnh báo, không phải lỗi); có nút *Kiểm tra lại link*. Chạy lại cell 14 trả về cùng link. Nếu tunnel lỗi, Run All vẫn tiếp tục và UI notebook vẫn dùng được.

## Cách chạy

1. Cách khuyến nghị: trước khi push, chạy `scripts/configure_repository.py --repo-url https://github.com/<owner>/<repo>.git` để cell 2 có sẵn `GITHUB_REPO_URL`. Nếu cell 2 vẫn còn `<OWNER>/<REPO>`, cell 3 tái sử dụng checkout có sẵn của dự án tại `TP_REPO_DIR`, hoặc hỏi URL **một lần** (không bao giờ tự đoán URL). `GIT_REF`: demo chính thức nên dùng commit SHA hoặc release tag. Chọn `DEMO_MODE`.
2. *Runtime → Run all*. Lần đầu, Colab hiện hộp thoại một lần “Warning: This notebook was not authored by Google” (notebook không do Google tạo) — bấm **Run anyway** để tiếp tục. Lần đầu bootstrap cần Internet (~1 GB tải về, build pyspark vài phút); các lần sau dùng cache đã kiểm checksum.
3. Sau khi restart kernel: chạy lại **cell 2**, rồi chạy tiếp cell cần dùng — dịch vụ nền vẫn chạy và được tái sử dụng.
4. Không cần GitHub token (repo public), API key MBTA, GPU, Docker, ngrok, tài khoản Cloudflare hay database cloud. Không đặt secret vào notebook.
5. Khi xong: dùng các nút ở cell 18 (dữ liệu, checkpoint, inputs được giữ lại trong `TP_WORKDIR`).

Dữ liệu nguồn: Massachusetts Bay Transportation Authority (MBTA) GTFS/GTFS-Realtime. Dự án học tập, không liên kết với MBTA.

In [ ]:
# [2/18] Tham số demo (cell có tag parameters; scripts/configure_repository.py cập nhật hai dòng repo/ref).
# Chỉ sửa khối giá trị ngay dưới đây. Không đặt token/API key vào notebook.
GITHUB_REPO_URL = "https://github.com/nqMinh-GR/de_btl.git"
GIT_REF = "main"              # Khi demo chính thức, điền commit SHA (40 ký tự hex) hoặc release tag.
DEMO_MODE = "synthetic"       # synthetic | replay | live
TP_REPO_DIR = "/content/transitpulse-mbta"
TP_WORKDIR = "/content/transitpulse-runtime"

ROUTE_IDS = []               # Live: chọn/xác minh 5 bus routes trước khi tạo run.
RECORDING_MANIFEST = ""      # Bắt buộc khi chọn replay.
REPLAY_SPEED = 10.0
SYNTHETIC_SEED = 42
DEMO_DURATION_SECONDS = 600  # Wall-time budget; không nhầm với recording/source duration.

RUN_UNIT_TESTS = True
RUN_GOLDEN_SMOKE = True
RUN_BENCHMARK = False
RUN_INTEGRATION_TESTS = False  # Tùy chọn: integration tests trên Kafka/Spark/Cassandra thật (lâu, tốn RAM).
EXPORT_ARTIFACTS = True
STOP_AFTER_RUN_ALL = False   # Giữ UI tương tác; cung cấp nút Stop riêng.
MOUNT_DRIVE = False          # Tùy chọn sao lưu ZIP, không làm working database.
STRICT_RUN_ALL = False       # True: cell 18 báo lỗi nếu còn bước FAIL. False: Run All chạy hết, lỗi vẫn hiện đỏ.

ENABLE_CLOUDFLARE_TUNNEL = True  # Tạo link công khai https://*.trycloudflare.com tới dashboard CHỈ ĐỌC, không cần
                                 # tài khoản/token Cloudflare. AI CÓ LINK ĐỀU XEM ĐƯỢC khi session còn chạy.
TUNNEL_TARGET = "dashboard"      # dashboard | api | both (api/both công khai cả API chỉ đọc)

# ---------------------------------------------------------------------------------------------------
# Phần dưới: kiểm tra tham số + hàm _ctx() dùng chung cho mọi cell (không cần sửa).
# Sau khi restart kernel: chạy lại cell này rồi chạy tiếp cell cần dùng (dịch vụ nền vẫn chạy).
import importlib.util
import json
import os
import re
import sys
from pathlib import Path
from types import SimpleNamespace

TP_PROFILE = 'colab-native'
_MODES = ('synthetic', 'replay', 'live')
_TUNNEL_TARGETS = ('dashboard', 'api', 'both')
_ROUTE_ID_RE = '[A-Za-z0-9._-]{1,64}'
_GIT_REF_RE = '[A-Za-z0-9][A-Za-z0-9._/-]{0,199}'


class ParameterError(ValueError):
    '''Tham số không hợp lệ: notebook dừng ngay, không tự đổi mode.'''


def _params():
    '''Đọc và kiểm tra tham số (gọi lại ở mọi cell nên giá trị papermill/sửa tay đều được kiểm tra).'''
    g = globals()
    errors, notes = [], []
    mode = str(g.get('DEMO_MODE', '')).strip().lower()
    if mode not in _MODES:
        errors.append('DEMO_MODE phải là một trong %s (đang là %r).' % (', '.join(_MODES), g.get('DEMO_MODE')))
    repo_url = str(g.get('GITHUB_REPO_URL', '')).strip()
    if not repo_url.startswith('https://'):
        errors.append('GITHUB_REPO_URL phải là URL https (không chứa token).')
    ref = str(g.get('GIT_REF', '')).strip()
    if not re.fullmatch(_GIT_REF_RE, ref) or '..' in ref:
        errors.append('GIT_REF không hợp lệ: %r.' % ref)
    repo_dir = str(g.get('TP_REPO_DIR', '')).strip()
    workdir = str(g.get('TP_WORKDIR', '')).strip()
    for name, value in (('TP_REPO_DIR', repo_dir), ('TP_WORKDIR', workdir)):
        if not os.path.isabs(value) or os.path.normpath(value) in ('/', '/content', '/content/drive'):
            errors.append('%s phải là đường dẫn tuyệt đối riêng (đang là %r).' % (name, value))
    if repo_dir and workdir and os.path.normpath(repo_dir) == os.path.normpath(workdir):
        errors.append('TP_REPO_DIR và TP_WORKDIR phải khác nhau.')
    if workdir.startswith('/content/drive'):
        errors.append('TP_WORKDIR không được nằm trên Google Drive (Kafka/Cassandra/checkpoint cần disk local).')
    routes = g.get('ROUTE_IDS') or []
    if isinstance(routes, str):
        routes = [r.strip() for r in routes.split(',') if r.strip()]
    if not isinstance(routes, (list, tuple)):
        errors.append('ROUTE_IDS phải là list các route_id.')
        routes = []
    routes = [str(r).strip() for r in routes]
    bad = [r for r in routes if not re.fullmatch(_ROUTE_ID_RE, r)]
    if bad:
        errors.append('route_id không hợp lệ: %s' % bad)
    if len(set(routes)) != len(routes):
        errors.append('ROUTE_IDS có phần tử trùng.')
    if len(routes) > 5:
        errors.append('Tối đa 5 route trong scope demo (đang có %d).' % len(routes))
    manifest = str(g.get('RECORDING_MANIFEST') or '').strip()
    if manifest and not os.path.isabs(manifest):
        manifest = os.path.join(repo_dir, manifest)
    try:
        speed = float(g.get('REPLAY_SPEED'))
    except (TypeError, ValueError):
        speed = float('nan')
    if not (0 < speed <= 100):
        errors.append('REPLAY_SPEED phải trong (0, 100] (đang là %r).' % g.get('REPLAY_SPEED'))
    seed = g.get('SYNTHETIC_SEED')
    if isinstance(seed, bool) or not isinstance(seed, int) or seed < 0:
        errors.append('SYNTHETIC_SEED phải là số nguyên >= 0.')
    duration = g.get('DEMO_DURATION_SECONDS')
    if isinstance(duration, bool) or not isinstance(duration, int) or not (60 <= duration <= 7200):
        errors.append('DEMO_DURATION_SECONDS phải là số nguyên trong [60, 7200].')
    flags = {}
    for name in ('RUN_UNIT_TESTS', 'RUN_GOLDEN_SMOKE', 'RUN_BENCHMARK', 'RUN_INTEGRATION_TESTS',
                 'EXPORT_ARTIFACTS', 'STOP_AFTER_RUN_ALL', 'MOUNT_DRIVE', 'STRICT_RUN_ALL',
                 'ENABLE_CLOUDFLARE_TUNNEL'):
        value = g.get(name, False)
        if not isinstance(value, bool):
            errors.append('%s phải là True/False.' % name)
        flags[name.lower()] = bool(value)
    tunnel_target = str(g.get('TUNNEL_TARGET', 'dashboard')).strip().lower()
    if tunnel_target not in _TUNNEL_TARGETS:
        errors.append('TUNNEL_TARGET phải là một trong %s (đang là %r).' % (', '.join(_TUNNEL_TARGETS),
                                                                           g.get('TUNNEL_TARGET')))
    if mode == 'replay' and not manifest:
        errors.append('Mode replay cần RECORDING_MANIFEST = đường dẫn tới recording_manifest.json của một bundle '
                      'đã ghi (collector record, hoặc ZIP artifact đã giải nén vào TP_WORKDIR/data/inputs). '
                      'Notebook không tự sinh recording MBTA giả và không tự chuyển sang synthetic.')
    if mode == 'live' and not routes:
        errors.append('Mode live cần ROUTE_IDS (1-5 bus route đã kiểm tra bằng source-check). '
                      'Notebook không tự chọn tuyến và không chạy synthetic dưới nhãn LIVE.')
    if mode != 'replay' and manifest:
        notes.append('RECORDING_MANIFEST bị bỏ qua vì DEMO_MODE=%s.' % mode)
    if mode == 'synthetic' and routes:
        notes.append('ROUTE_IDS bị bỏ qua ở synthetic (fixture có 5 tuyến synthetic riêng).')
    if flags.get('enable_cloudflare_tunnel'):
        notes.append('Tunnel bật: link https://*.trycloudflare.com là CÔNG KHAI, ai có link đều xem được (%s).'
                     % tunnel_target)
    if errors:
        raise ParameterError('Tham số không hợp lệ:' + ''.join(chr(10) + ' - ' + e for e in errors))
    return SimpleNamespace(mode=mode, repo_url=repo_url, git_ref=ref, repo_dir=repo_dir, workdir=workdir,
                           route_ids=routes, manifest=manifest, speed=speed, seed=seed, duration=duration,
                           tunnel_target=tunnel_target, notes=notes, **flags)


def _load_ui(repo_dir=None):
    '''Nạp (lại) notebooks/colab_ui.py từ repo đã clone; an toàn sau restart kernel.'''
    path = Path(repo_dir or _params().repo_dir) / 'notebooks' / 'colab_ui.py'
    if not path.is_file():
        raise FileNotFoundError('Không thấy %s — chạy cell 3 (clone) trước.' % path)
    name = 'transitpulse_colab_ui'
    spec = importlib.util.spec_from_file_location(name, str(path))
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


def _ctx():
    '''Ngữ cảnh của mọi cell sau bootstrap: tham số đã kiểm tra, runtime.json, colab_ui, session gần nhất.'''
    p = _params()
    ui = _load_ui(p.repo_dir)
    rt = ui.load_runtime(p.workdir)
    for key, expected in (('repo_dir', p.repo_dir), ('workdir', p.workdir)):
        if os.path.realpath(str(rt.get(key))) != os.path.realpath(expected):
            raise ui.CellFailed('RUNTIME_MISMATCH', 'runtime.json có %s=%s, khác tham số %s.' % (key, rt.get(key), expected),
                                hint='Chạy lại cell 4 (bootstrap) cho repo/workdir hiện tại.')
    work = Path(str(rt.get('workdir')))
    start = ui.read_json(work / 'processes' / 'last_start.json')
    api_base = (((start or {}).get('api') or {}).get('base_url') or (rt.get('api') or {}).get('base_url')
                or ui.DEFAULT_API_BASE)

    def run_cli(script, *args, **kwargs):
        return ui.run_cli(rt, script, *args, **kwargs)

    def require_start():
        if not isinstance(start, dict) or not start.get('run_id') or not start.get('session_id'):
            raise ui.CellFailed('NO_SESSION', 'Chưa có session demo (TP_WORKDIR/processes/last_start.json).',
                                hint='Chạy cell 11-13 (prepare, start, wait-ready) trước.')
        return start

    return SimpleNamespace(p=p, ui=ui, rt=rt, repo=Path(str(rt.get('repo_dir'))), work=work,
                           core_python=rt['python']['core_python'], profile=rt.get('profile') or TP_PROFILE,
                           start=start, api_base=api_base, run_cli=run_cli, require_start=require_start)


_P = _params()
print('Tham số hợp lệ: mode=%s | ref=%s | repo=%s | workdir=%s' % (_P.mode, _P.git_ref, _P.repo_dir, _P.workdir))
print('Python kernel: %s (core Python 3.11 nằm trong .venv của repo sau bootstrap)' % sys.version.split()[0])
for _msg in _P.notes:
    print('Lưu ý:', _msg)

In [ ]:
# [3/18] Clone/checkout đúng repository và ref. Chạy lại an toàn: tái sử dụng clone sạch đúng remote/ref;
# không bao giờ reset cứng hay xóa repo để che khác biệt, không đổi commit khi một demo session đang chạy.
# GITHUB_REPO_URL còn placeholder <OWNER>/<REPO> (chưa chạy scripts/configure_repository.py trước khi push):
# tái sử dụng checkout có sẵn của dự án tại TP_REPO_DIR, nếu không có thì hỏi URL MỘT lần. Không bao giờ tự đoán URL.
# Tùy chọn nâng cao (gán trong cell này hoặc cell 2 trước khi chạy):
#   REFRESH_REF = True   -> fetch và checkout lại GIT_REF dạng nhánh/tag (working tree phải sạch)
#   PRIVATE_REPO = True  -> repo private: token đọc từ Colab Secrets (GITHUB_TOKEN) hoặc getpass, chỉ nằm trong
#                           env của tiến trình git (askpass tạm 0700, xóa ngay), không lưu vào remote/notebook.
import getpass
import shutil
import subprocess
import tempfile
import urllib.parse

P = _params()
REFRESH_REF = bool(globals().get('REFRESH_REF', False))
PRIVATE_REPO = bool(globals().get('PRIVATE_REPO', False))
REPO = Path(P.repo_dir)
IS_SHA = re.fullmatch('[0-9a-f]{40}', P.git_ref) is not None
_SEG_RE = '[A-Za-z0-9_.-]{1,100}'


def _github_url(url):
    '''https://github.com/<owner>/<repo>(.git) -> https://github.com/<owner>/<repo>.git; None nếu sai/có credentials.'''
    url = (url or '').strip()
    try:
        parts = urllib.parse.urlsplit(url)
        port = parts.port
    except ValueError:
        return None
    if parts.scheme != 'https' or (parts.hostname or '').lower() != 'github.com':
        return None
    if parts.username or parts.password or '@' in parts.netloc or port is not None or parts.query or parts.fragment:
        return None
    segs = [s for s in parts.path.split('/') if s]
    if len(segs) != 2:
        return None
    owner = segs[0]
    name = segs[1][:-4] if segs[1].endswith('.git') else segs[1]
    for seg in (owner, name):
        if not re.fullmatch(_SEG_RE, seg) or seg.strip('.') == '':
            return None
    return 'https://github.com/%s/%s.git' % (owner, name)


def _is_project_checkout(path):
    return ((path / '.git').exists() and (path / 'notebooks' / 'colab_ui.py').is_file()
            and (path / 'scripts' / 'bootstrap_colab.sh').is_file())


REPO_URL = P.repo_url
if '<' in REPO_URL or '>' in REPO_URL:
    if _is_project_checkout(REPO):
        _origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], stdout=subprocess.PIPE,
                                 stderr=subprocess.DEVNULL, text=True).stdout.strip()
        REPO_URL = _github_url(_origin)
        if REPO_URL is None:
            raise ParameterError('GITHUB_REPO_URL còn placeholder và origin của %s không phải URL '
                                 'https://github.com/<owner>/<repo> hợp lệ (không chứa thông tin đăng nhập). '
                                 'Sửa GITHUB_REPO_URL ở cell 2.' % REPO)
        print('GITHUB_REPO_URL còn placeholder: tái sử dụng checkout có sẵn tại', REPO, 'với origin', REPO_URL)
    else:
        print('GITHUB_REPO_URL còn placeholder <OWNER>/<REPO>. Cách khuyến nghị: chạy scripts/configure_repository.py '
              '--repo-url ... trước khi push để Run All không phải hỏi.')
        try:
            _answer = input('Nhập URL GitHub của repo (https://github.com/<owner>/<repo>.git): ')
        except Exception as exc:
            raise ParameterError('Không đọc được URL nhập vào (%s). Sửa GITHUB_REPO_URL ở cell 2 rồi chạy lại.'
                                 % type(exc).__name__)
        REPO_URL = _github_url(_answer)
        if REPO_URL is None:
            raise ParameterError('URL không hợp lệ: cần dạng https://github.com/<owner>/<repo>(.git), không chứa '
                                 'token/mật khẩu. Notebook không tự đoán URL; sửa GITHUB_REPO_URL ở cell 2 hoặc chạy lại cell này.')
    GITHUB_REPO_URL = REPO_URL  # các cell sau dùng URL đã xác nhận (chỉ trong kernel, không ghi vào file)
else:
    _url = urllib.parse.urlsplit(REPO_URL)
    if _url.scheme != 'https' or not _url.hostname or _url.username or _url.password or '@' in _url.netloc:
        raise ParameterError('GITHUB_REPO_URL phải là https và không chứa thông tin đăng nhập/token.')
    if _url.hostname.lower() != 'github.com':
        print('CẢNH BÁO: repo không nằm trên github.com:', _url.hostname)


def _norm_url(url):
    url = url.strip().rstrip('/')
    if url.endswith('.git'):
        url = url[:-4]
    parts = urllib.parse.urlsplit(url)
    return ('%s://%s%s' % (parts.scheme, parts.hostname or '', parts.path)).lower()


def _git(*args, env=None, timeout=900):
    cmd = ['git', '-c', 'credential.helper=', *args]
    r = subprocess.run(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, timeout=timeout)
    if r.returncode != 0:
        raise RuntimeError('git %s thất bại (exit %d): %s' % (' '.join(args[:4]), r.returncode, r.stderr.strip()[-2000:]))
    return r.stdout.strip()


def _git_ok(*args):
    return subprocess.run(['git', *args], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0


def _git_env():
    env = dict(os.environ)
    env['GIT_TERMINAL_PROMPT'] = '0'
    if not PRIVATE_REPO:
        return env, None
    token = None
    try:
        from google.colab import userdata
        token = userdata.get('GITHUB_TOKEN')
    except Exception:
        token = None
    if not token:
        token = getpass.getpass('GitHub token (chỉ dùng cho tiến trình git này, không lưu): ')
    tmp = tempfile.mkdtemp(prefix='tp-askpass-')
    script = Path(tmp) / 'askpass.sh'
    script.write_text(chr(10).join(['#!/bin/sh', 'case $1 in', '  *sername*) echo x-access-token ;;',
                                    '  *) echo $TP_GIT_TOKEN ;;', 'esac', '']))
    os.chmod(script, 0o700)
    env['GIT_ASKPASS'] = str(script)
    env['TP_GIT_TOKEN'] = token
    return env, tmp


def _active_session():
    reg_path = Path(P.workdir) / 'processes' / 'registry.json'
    try:
        reg = json.loads(reg_path.read_text(encoding='utf-8'))
    except FileNotFoundError:
        return None
    except (OSError, ValueError):
        return 'không đọc được %s' % reg_path
    sid = reg.get('current_session')
    if not sid:
        return None
    state = None
    try:
        session_json = Path(P.workdir) / 'sessions' / str(sid) / 'session.json'
        state = json.loads(session_json.read_text(encoding='utf-8')).get('state')
    except (OSError, ValueError):
        state = None
    if state in ('STOPPED', 'FAILED'):
        return None
    return '%s (state=%s)' % (sid, state)


def _require_no_session(action):
    active = _active_session()
    if active:
        raise RuntimeError('SESSION_ACTIVE: không %s khi demo session %s đang chạy. Dừng session (cell 18 hoặc '
                           'demo.py stop) rồi chạy lại cell này.' % (action, active))


_env, _askpass_dir = _git_env()
action = 'REUSED'
try:
    if not REPO.exists():
        REPO.parent.mkdir(parents=True, exist_ok=True)
        if IS_SHA:
            _git('clone', '--no-checkout', REPO_URL, str(REPO), env=_env)
            _git('-C', str(REPO), 'checkout', '--detach', P.git_ref)
        else:
            _git('clone', '--branch', P.git_ref, '--single-branch', REPO_URL, str(REPO), env=_env)
        action = 'CLONED'
    else:
        if not (REPO / '.git').exists():
            raise RuntimeError('REPO_NOT_GIT: %s đã tồn tại nhưng không phải git repo. Notebook không xóa thư mục; '
                               'đổi TP_REPO_DIR hoặc tự di chuyển thư mục.' % REPO)
        origin = _git('-C', str(REPO), 'remote', 'get-url', 'origin')
        if _norm_url(origin) != _norm_url(REPO_URL):
            raise RuntimeError('REPO_REMOTE_MISMATCH: origin của %s khác GITHUB_REPO_URL=%s. Không ghi đè; '
                               'đổi TP_REPO_DIR hoặc sửa tham số.' % (REPO, REPO_URL))
        dirty = _git('-C', str(REPO), 'status', '--porcelain', '--untracked-files=no')
        if dirty:
            raise RuntimeError('REPO_DIRTY: working tree có thay đổi chưa commit:' + chr(10) + dirty + chr(10)
                               + 'Tự commit/stash/khôi phục; notebook không reset cứng hay ghi đè thay đổi.')
        head = _git('-C', str(REPO), 'rev-parse', 'HEAD')
        if IS_SHA:
            if head != P.git_ref:
                _require_no_session('đổi commit')
                if not _git_ok('-C', str(REPO), 'cat-file', '-e', P.git_ref + '^{commit}'):
                    _git('-C', str(REPO), 'fetch', 'origin', P.git_ref, env=_env)
                _git('-C', str(REPO), 'checkout', '--detach', P.git_ref)
                action = 'CHECKED_OUT'
        else:
            branch = _git('-C', str(REPO), 'rev-parse', '--abbrev-ref', 'HEAD')
            tags = _git('-C', str(REPO), 'tag', '--points-at', 'HEAD').split()
            on_ref = branch == P.git_ref or P.git_ref in tags
            if REFRESH_REF:
                _require_no_session('fetch/checkout lại ref')
                is_tag = bool(_git('-C', str(REPO), 'ls-remote', '--tags', 'origin', 'refs/tags/' + P.git_ref, env=_env))
                if is_tag:
                    _git('-C', str(REPO), 'fetch', 'origin', 'refs/tags/%s:refs/tags/%s' % (P.git_ref, P.git_ref), env=_env)
                    _git('-C', str(REPO), 'checkout', '--detach', 'refs/tags/' + P.git_ref)
                else:
                    _git('-C', str(REPO), 'fetch', 'origin', P.git_ref, env=_env)
                    if _git_ok('-C', str(REPO), 'show-ref', '--verify', '--quiet', 'refs/heads/' + P.git_ref):
                        _git('-C', str(REPO), 'checkout', P.git_ref)
                        _git('-C', str(REPO), 'merge', '--ff-only', 'FETCH_HEAD')
                    else:
                        _git('-C', str(REPO), 'checkout', '-b', P.git_ref, 'FETCH_HEAD')
                action = 'REFRESHED'
            elif not on_ref:
                raise RuntimeError('REF_MISMATCH: repo đang ở %s (%s), khác GIT_REF=%s. Đặt REFRESH_REF = True rồi '
                                   'chạy lại cell này (cần working tree sạch và không có session đang chạy).'
                                   % (branch, head[:12], P.git_ref))
finally:
    if _askpass_dir:
        shutil.rmtree(_askpass_dir, ignore_errors=True)
    _env = None

TP_REPO_COMMIT = _git('-C', str(REPO), 'rev-parse', 'HEAD')
_branch = _git('-C', str(REPO), 'rev-parse', '--abbrev-ref', 'HEAD')
_status = _git('-C', str(REPO), 'status', '--porcelain', '--untracked-files=normal')
_untracked = [line[3:] for line in _status.splitlines() if line.startswith('??')]
print('Repo        :', REPO, '(' + action + ')')
print('Remote      :', REPO_URL)
print('GIT_REF     :', P.git_ref, '| HEAD:', _branch)
print('Commit SHA  :', TP_REPO_COMMIT)
print('Working tree: sạch (không có thay đổi tracked)')
if _untracked:
    print('WARN untracked (uv.lock sinh trên Colab là bình thường):', ', '.join(_untracked[:20]))
if IS_SHA:
    print('Commit khớp GIT_REF.' if TP_REPO_COMMIT == P.git_ref else 'CẢNH BÁO: commit khác GIT_REF!')
else:
    print('Lưu ý: GIT_REF là nhánh/tag nên có thể trôi; notebook đang mở có thể thuộc commit khác commit vừa clone. '
          'Bản demo chính thức nên mở notebook và clone cùng một commit SHA hoặc release tag. '
          'Mọi report ghi SHA ở trên.')

In [ ]:
# [4/18] Bootstrap native runtime: uv + Python 3.11 (.venv), JDK 17, Kafka, Cassandra, Spark connector JARs,
# dashboard extra (Streamlit/PyDeck, luôn được cài) và cloudflared. ENABLE_CLOUDFLARE_TUNNEL=False thêm --no-tunnel:
# chỉ bỏ qua cloudflared. Lần đầu tải khoảng 1 GB và build pyspark (vài phút); chạy lại sẽ tái sử dụng cache đã
# kiểm checksum.
P = _params()
ui = _load_ui(P.repo_dir)
bootstrap_argv = ['/bin/bash', f'{TP_REPO_DIR}/scripts/bootstrap_colab.sh', '--workdir', TP_WORKDIR]
if not P.enable_cloudflare_tunnel:
    bootstrap_argv.append('--no-tunnel')
rc, _out, tail, secs = ui.stream_command(bootstrap_argv, cwd=P.repo_dir, env=ui.bootstrap_env(), timeout=1800,
                                         label='bootstrap_colab.sh')
ui.show_bootstrap_report(ui.read_json(Path(P.workdir) / 'runtime' / 'bootstrap_report.json'))
if rc != 0:
    ui.record_step(P.workdir, 'bootstrap', 'FAIL', exit_code=rc, no_tunnel=not P.enable_cloudflare_tunnel)
    raise ui.CellFailed('BOOTSTRAP_EXIT_%d' % rc, 'bootstrap_colab.sh thất bại.',
                        hint='Đọc bước FAIL trong bootstrap report và log phía trên; sửa nguyên nhân rồi chạy lại '
                             '(cache đã tải được giữ lại).',
                        log_excerpt=chr(10).join(tail[-60:]), exit_code=rc)
rt = ui.load_runtime(P.workdir)
_py, _java, _spark = rt.get('python') or {}, rt.get('java') or {}, rt.get('spark') or {}
ui.show_kv({
    'core_python': _py.get('core_python'), 'python': _py.get('version'), 'lock_origin': _py.get('lock_origin'),
    'java': '%s %s' % (_java.get('vendor'), _java.get('version')), 'spark': _spark.get('version'),
    'connector_jars': len(_spark.get('connector_jars') or []), 'kafka_home': (rt.get('kafka') or {}).get('home'),
    'cassandra_home': (rt.get('cassandra') or {}).get('home'), 'repo_commit': rt.get('repo_commit'),
    'api': (rt.get('api') or {}).get('base_url'),
    'cloudflared': 'được cài' if P.enable_cloudflare_tunnel else 'bỏ qua (--no-tunnel)',
}, title='runtime.json')
if _py.get('lock_origin') == 'generated_on_colab':
    print('uv.lock được sinh trên Colab (lock_origin=generated_on_colab); file này có trong evidence ZIP để commit lại.')
ui.record_step(P.workdir, 'bootstrap', 'PASS', seconds=round(secs, 1), lock_origin=_py.get('lock_origin'),
               no_tunnel=not P.enable_cloudflare_tunnel)

In [ ]:
ctx = _ctx()  # [5/18] Kernel UI nhẹ: kernel Colab chỉ cần ipywidgets; core (Spark/Kafka/Cassandra) ở .venv.
# Chỉ cài requirements-kernel.txt khi import ipywidgets thất bại; không nâng cấp gói có sẵn của Colab.
import platform

print('Kernel Python :', platform.python_version(), '|', sys.executable)
print('Core Python   :', (ctx.rt.get('python') or {}).get('version'), '|', ctx.core_python)
try:
    import ipywidgets
    print('ipywidgets    :', ipywidgets.__version__, '(có sẵn, không cài/nâng cấp)')
    widgets_status = 'PRESENT'
except ImportError:
    req = ctx.repo / 'notebooks' / 'requirements-kernel.txt'
    rc, _out, tail, _secs = ctx.ui.stream_command([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(req)],
                                                  timeout=600, label='pip install ipywidgets (kernel)')
    widgets_status = 'INSTALLED' if rc == 0 else 'UNAVAILABLE'
    if rc != 0:
        print('CẢNH BÁO: không cài được ipywidgets; UI sẽ dùng HTML tĩnh (vẫn chỉ gọi API).')
    importlib.invalidate_caches()
ctx = _ctx()  # nạp lại colab_ui để nhận ipywidgets nếu vừa cài
print('Widgets trong colab_ui:', 'có' if ctx.ui.widgets is not None else 'không (fallback HTML tĩnh)')
ctx.ui.record_step(ctx.work, 'kernel_ui', widgets_status, kernel_python=platform.python_version(),
                   core_python=(ctx.rt.get('python') or {}).get('version'))

In [ ]:
ctx = _ctx()  # [6/18] Doctor / resource preflight: CPU, RAM khả dụng, disk, ports, hash config/lock, Java/.venv.
import platform

try:
    doctor = ctx.run_cli('doctor.py', '--profile', ctx.profile, '--kernel-python', platform.python_version(),
                         timeout=300, label='doctor',
                         show=lambda d: ctx.ui.show_cli_result(d, 'Doctor / preflight', ('overall', 'profile')))
except ctx.ui.CellFailed as exc:
    ctx.ui.record_step(ctx.work, 'doctor', 'FAIL', code=exc.code)
    raise
ctx.ui.record_step(ctx.work, 'doctor', 'PASS')

In [ ]:
ctx = _ctx()  # [7/18] Start Kafka (KRaft) + Cassandra native, rồi init topics + CQL migrations (idempotent).
# Chạy lại an toàn: supervisor tái sử dụng broker/node đang chạy (một broker, một node).
infra_start = ctx.run_cli('infra.py', 'start', '--profile', ctx.profile, timeout=900, label='infra start',
                          show=lambda d: ctx.ui.show_cli_result(d, 'infra start (Kafka + Cassandra)', ('reused',)))
infra_init = ctx.run_cli('infra.py', 'init', '--profile', ctx.profile, timeout=900, label='infra init',
                         show=lambda d: ctx.ui.show_cli_result(d, 'infra init (topics + CQL)', ('cluster_id', 'release_version')))
ctx.ui.record_step(ctx.work, 'infra', 'PASS')

In [ ]:
ctx = _ctx()  # [8/18] Unit + runtime + Spark tests nếu RUN_UNIT_TESTS. Kết quả thật, không bị nuốt.
# Test lỗi: ghi FAIL (kèm output và đường dẫn junit), hiện khung đỏ, Run All vẫn tiếp tục để session/UI/link vẫn
# dùng được; tóm tắt đỏ ở cell 14/17/18, STRICT_RUN_ALL=True làm cell 18 báo lỗi.
# tests/spark là các vector kiểm thử mức unit (SparkSession local[1], không Kafka/Cassandra): KHÔNG phải bằng chứng
# integration. Integration tests trên hạ tầng thật nằm ở cell 10 (RUN_INTEGRATION_TESTS).
if ctx.p.run_unit_tests:
    unit = ctx.ui.run_pytest(ctx.rt, ['tests/unit', 'tests/runtime', 'tests/spark'], marker='unit or runtime or spark',
                             report_name='unit', timeout=5400, check=False,
                             extra_args=['--timeout=300'])
    if unit['status'] == 'PASS':
        print('Unit/runtime/Spark tests: PASS | junit:', unit['junit'])
    else:
        print('Unit/runtime/Spark tests: FAIL (xem khung đỏ phía trên) | junit:', unit['junit'])
    print('Lưu ý: đây là bằng chứng mức unit, không thay cho integration tests (cell 10).')
else:
    ctx.ui.record_step(ctx.work, 'pytest_unit', 'NOT_RUN', reason='RUN_UNIT_TESTS=False')
    print('NOT_RUN: RUN_UNIT_TESTS=False')

In [ ]:
ctx = _ctx()  # [9/18] Golden full-stack smoke: input cố định -> Kafka -> Spark -> Cassandra -> API, so với oracle.
# Chạy trên run riêng và dừng processor của run đó trước demo tương tác (một Spark JVM tại một thời điểm).
# Nếu demo session trước còn RUNNING/PAUSED (ví dụ Run All lần 2) thì ghi NOT_RUN kèm gợi ý.
# Golden smoke lỗi: ghi FAIL + khung đỏ, Run All vẫn tiếp tục (tóm tắt ở cell 14/17/18).
activity = ctx.ui.session_activity(ctx.work)
if not ctx.p.run_golden_smoke:
    ctx.ui.record_step(ctx.work, 'golden_smoke', 'NOT_RUN', reason='RUN_GOLDEN_SMOKE=False')
    print('NOT_RUN: RUN_GOLDEN_SMOKE=False')
elif activity['busy']:
    ctx.ui.record_step(ctx.work, 'golden_smoke', 'NOT_RUN', reason='SESSION_ACTIVE', detail=activity['reason'],
                       session_id=activity['session_id'], hint=ctx.ui.SESSION_BUSY_HINT)
    print('Golden smoke: NOT_RUN (bỏ qua) vì ' + str(activity['reason']) + '; golden smoke cần slot processor riêng.')
    print('Gợi ý: ' + ctx.ui.SESSION_BUSY_HINT)
else:
    try:
        golden = ctx.run_cli('demo.py', 'golden-smoke', '--timeout-seconds', '900', timeout=1200, label='golden-smoke',
                             show=lambda d: ctx.ui.show_cli_result(d, 'Golden smoke', ('status', 'run_id', 'report_path')))
        golden_status = str(golden.get('status') or '').upper()
        if golden_status in ('FAIL', 'FAILED', 'ERROR') or golden.get('passed') is False:
            raise ctx.ui.CellFailed('GOLDEN_SMOKE_FAILED', 'golden-smoke trả về status=%s.' % (golden.get('status'),),
                                    hint='Xem bảng checks phía trên và report_path.', result=golden)
    except ctx.ui.CellFailed as exc:
        ctx.ui.record_failure(ctx.work, 'golden_smoke', exc)
        print('Golden smoke: FAIL (xem khung đỏ phía trên).')
    else:
        ctx.ui.record_step(ctx.work, 'golden_smoke', 'PASS', run_id=golden.get('run_id'),
                           report_path=golden.get('report_path'), cli_status=golden.get('status'))
        print('Golden smoke: PASS | run', golden.get('run_id'))

In [ ]:
ctx = _ctx()  # [10/18] Tùy chọn: integration tests (RUN_INTEGRATION_TESTS) và benchmark có giới hạn (RUN_BENCHMARK).
# Cả hai chạy trên run riêng TRƯỚC demo tương tác; mặc định NOT_RUN (Run All không bật stress nặng).
# Nếu demo session còn RUNNING/PAUSED thì ghi NOT_RUN kèm gợi ý dừng session. Lỗi: ghi FAIL + khung đỏ, Run All
# vẫn tiếp tục (tóm tắt ở cell 14/17/18). Trạng thái benchmark lấy từ statuses (MEASURED > PARTIAL > NOT_RUN),
# không bao giờ mặc định là MEASURED.
activity = ctx.ui.session_activity(ctx.work)
if not ctx.p.run_integration_tests:
    ctx.ui.record_step(ctx.work, 'pytest_integration', 'NOT_RUN', reason='RUN_INTEGRATION_TESTS=False')
    print('Integration tests: NOT_RUN (RUN_INTEGRATION_TESTS=False)')
elif activity['busy']:
    ctx.ui.record_step(ctx.work, 'pytest_integration', 'NOT_RUN', reason='SESSION_ACTIVE', detail=activity['reason'],
                       session_id=activity['session_id'], hint=ctx.ui.SESSION_BUSY_HINT)
    print('Integration tests: NOT_RUN (bỏ qua) vì ' + str(activity['reason']) + '. Gợi ý: ' + ctx.ui.SESSION_BUSY_HINT)
else:
    integ = ctx.ui.run_pytest(ctx.rt, ['tests/integration'], marker='integration', report_name='integration',
                              timeout=5400, check=False)
    if integ['status'] == 'PASS':
        print('Integration tests: PASS | junit:', integ['junit'])
    else:
        print('Integration tests: FAIL (xem khung đỏ phía trên) | junit:', integ['junit'])
if not ctx.p.run_benchmark:
    ctx.ui.record_step(ctx.work, 'benchmark', 'NOT_RUN', reason='RUN_BENCHMARK=False')
    print('Benchmark: NOT_RUN (RUN_BENCHMARK=False). Các mục tiêu trong tài liệu không phải kết quả đo.')
elif activity['busy']:
    ctx.ui.record_step(ctx.work, 'benchmark', 'NOT_RUN', reason='SESSION_ACTIVE', detail=activity['reason'],
                       session_id=activity['session_id'], hint=ctx.ui.SESSION_BUSY_HINT)
    print('Benchmark: NOT_RUN (bỏ qua) vì ' + str(activity['reason']) + '. Gợi ý: ' + ctx.ui.SESSION_BUSY_HINT)
else:
    bench_args = ['benchmark', '--max-seconds', '300']
    if ctx.p.mode == 'replay':
        bench_args += ['--scenario', 'replay_10x', '--recording-manifest', ctx.p.manifest]
    else:
        bench_args += ['--scenario', 'stress_100']
    try:
        bench = ctx.run_cli('demo.py', *bench_args, timeout=1200, label='benchmark',
                            show=lambda d: ctx.ui.show_cli_result(d, 'Benchmark (giới hạn 300 s)', ('bench_id',)))
    except ctx.ui.CellFailed as exc:
        ctx.ui.record_failure(ctx.work, 'benchmark', exc)
        print('Benchmark: FAIL (xem khung đỏ phía trên).')
    else:
        bench_status = ctx.ui.benchmark_status(bench)
        ctx.ui.record_step(ctx.work, 'benchmark', bench_status, bench_id=bench.get('bench_id'),
                           statuses=bench.get('statuses'))
        print('Benchmark:', bench_status, '| statuses:', bench.get('statuses'), '| bench_id:', bench.get('bench_id'))

In [ ]:
ctx = _ctx()  # [11/18] Chuẩn bị inputs đúng mode: static version, scope, checksum inputs và nguồn dữ liệu.
p = ctx.p
prep_args = ['prepare', '--mode', p.mode]
if p.mode == 'synthetic':
    prep_args += ['--seed', str(p.seed)]
elif p.mode == 'replay':
    if not os.path.isfile(p.manifest):
        raise ctx.ui.CellFailed('RECORDING_MANIFEST_NOT_FOUND', 'Không thấy %s.' % p.manifest,
                                hint='Cung cấp bundle replay (recording_manifest.json + raw + static ZIP đúng version). '
                                     'Notebook không tự sinh recording MBTA giả và không chuyển sang synthetic.')
    prep_args += ['--recording-manifest', p.manifest]
    if p.route_ids:
        prep_args += ['--route-ids', ','.join(p.route_ids)]
else:
    prep_args += ['--route-ids', ','.join(p.route_ids)]
prepared = ctx.run_cli('demo.py', *prep_args, timeout=1800, label='prepare (%s)' % p.mode,
                       show=lambda d: ctx.ui.show_cli_result(d, 'Inputs cho mode %s' % p.mode, ('gtfs_version', 'scope_id')))
ctx.ui.record_step(ctx.work, 'prepare', 'PASS', mode=p.mode)

In [ ]:
ctx = _ctx()  # [12/18] Start demo session (detached, supervisor quản lý; Run All không bị chặn).
# Thứ tự: processor (checkpoint theo run_id) -> publisher/collector -> API. Chạy lại: tái sử dụng session đang chạy.
p = ctx.p
start_args = ['start', '--profile', ctx.profile, '--mode', p.mode, '--duration-seconds', str(p.duration),
              '--detach', '--ready-timeout', '300']
if p.mode == 'synthetic':
    start_args += ['--seed', str(p.seed)]
elif p.mode == 'replay':
    start_args += ['--recording-manifest', p.manifest, '--speed', str(p.speed)]
    if p.route_ids:
        start_args += ['--route-ids', ','.join(p.route_ids)]
else:
    start_args += ['--route-ids', ','.join(p.route_ids)]
START = ctx.run_cli('demo.py', *start_args, timeout=900, label='demo start (%s)' % p.mode,
                    show=lambda d: ctx.ui.show_cli_result(d, 'Demo session', ('session_id', 'run_id', 'source_type',
                                                                              'mode', 'reused', 'deadline_at')))
# So sánh mode; source_type phải phù hợp mode. Replay một recording synthetic hợp lệ trả về mode=replay và
# source_type=synthetic (banner vẫn ghi SYNTHETIC). LIVE chỉ chấp nhận source_type=live.
got_mode = str(START.get('mode') or '').lower()
got_source = str(START.get('source_type') or '').lower()
allowed_sources = ('replay', 'synthetic') if p.mode == 'replay' else (p.mode,)
if (got_mode and got_mode != p.mode) or (got_source and got_source not in allowed_sources):
    raise ctx.ui.CellFailed('MODE_MISMATCH', 'Session trả về mode=%s, source_type=%s; DEMO_MODE=%s chỉ chấp nhận '
                            'source_type %s.' % (got_mode, got_source, p.mode, '/'.join(allowed_sources)),
                            hint='Không bao giờ tự đổi mode: dừng session cũ (cell 18) rồi start lại.')
if not START.get('run_id') or not START.get('session_id'):
    raise ctx.ui.CellFailed('START_JSON_INCOMPLETE', 'JSON của demo.py start thiếu run_id/session_id.')
ctx.ui.write_json_atomic(ctx.work / 'processes' / 'last_start.json', START)
ctx.ui.record_step(ctx.work, 'start', 'PASS', session_id=START.get('session_id'), run_id=START.get('run_id'),
                   mode=p.mode, source_type=got_source or None)
print('Session:', START.get('session_id'), '| Run:', START.get('run_id'), '| mode:', got_mode or p.mode,
      '| source_type:', got_source or '?', '| reused:', START.get('reused'))

In [ ]:
ctx = _ctx()  # [13/18] Chờ sẵn sàng (có giới hạn): Spark query progress, generation READY và API trả lời.
start = ctx.require_start()
ready = ctx.run_cli('demo.py', 'wait-ready', '--session-id', start['session_id'], '--timeout-seconds', '300',
                    timeout=420, label='wait-ready', show=lambda d: ctx.ui.show_cli_result(d, 'Sẵn sàng', ('ready', 'state')))
client = ctx.ui.ApiClient(ctx.api_base)
run_id = start['run_id']
ctx.ui.show_api_probe([
    ('GET /health/ready', client.get('/health/ready')),
    ('GET /runs/' + run_id, client.get('/runs/' + ctx.ui.seg(run_id))),
    ('GET /quality?run_id=' + run_id, client.get('/quality', {'run_id': run_id})),
])
ctx.ui.record_step(ctx.work, 'wait_ready', 'PASS', run_id=run_id)

In [ ]:
ctx = _ctx()  # [14/18] Tóm tắt lỗi + giao diện notebook 5 tab + (tùy chọn) link công khai Cloudflare chỉ đọc.
# Khung đỏ đầu cell liệt kê mọi bước đã FAIL (tests, golden smoke, benchmark...) trước khi hiện UI.
# UI notebook chỉ gọi FastAPI qua HTTP từ kernel. Nút Làm mới là cách chính; Tự làm mới có giới hạn (tối đa 60 lần).
# Chạy lại cell sẽ đóng UI cũ. ENABLE_CLOUDFLARE_TUNNEL: demo.py expose start mở URL https://*.trycloudflare.com
# (không cần tài khoản/token; chạy lại trả về cùng link). AI CÓ LINK ĐỀU XEM ĐƯỢC khi session còn chạy.
# Tunnel lỗi hoặc chưa truy cập được không làm dừng Run All; UI notebook vẫn dùng được.
ctx.ui.show_failure_summary(ctx.work, 'trước giao diện')
start = ctx.require_start()
ui = ctx.ui.TransitPulseUI(ctx.api_base, session=start, refresh_seconds=5)
ui.show()
if ctx.p.enable_cloudflare_tunnel:
    tunnel = ctx.ui.TunnelPanel(ctx.rt, target=ctx.p.tunnel_target)
    tunnel.start(ready_timeout=120)
else:
    # Tunnel tắt: đóng mọi link công khai còn sót từ lần chạy trước (expose stop không cần cloudflared).
    _doc, _err, _code = ctx.ui.expose_stop(ctx.rt, target='both', show=False)
    ctx.ui.record_step(ctx.work, 'tunnel', 'NOT_RUN', reason='ENABLE_CLOUDFLARE_TUNNEL=False')
    if _err:
        print('CẢNH BÁO: không đóng được tunnel cũ (xem khung đỏ phía trên): ' + _err)
    else:
        print('Tunnel: NOT_RUN (ENABLE_CLOUDFLARE_TUNNEL=False). Đã chạy demo.py expose stop --target both để đóng '
              'mọi link công khai còn sót từ lần chạy trước (nếu có). Dashboard chỉ truy cập được bên trong VM.')

In [ ]:
ctx = _ctx()  # [15/18] Evidence/lineage: snapshot -> raw sha256 -> Kafka offset -> batch -> Cassandra READY -> API.
# Cell này chỉ minh họa truy vết (CLI đọc raw/offsets để giải thích); UI ứng dụng vẫn chỉ dùng API.
start = ctx.require_start()
lineage = ctx.run_cli('demo.py', 'lineage', '--run-id', start['run_id'], timeout=300, label='lineage',
                      show=lambda d: ctx.ui.show_cli_result(d, 'Lineage', ('run_id', 'feed', 'snapshot_id', 'batch_id')))
pointers = ctx.ui.ApiClient(ctx.api_base).get('/quality', {'run_id': start['run_id']})
ctx.ui.show_html('<h4>Con trỏ hiện tại theo API (meta.feeds)</h4>' + ctx.ui.feed_health_table(pointers.meta))
print('Nếu snapshot_id trong lineage khác con trỏ hiện tại: pipeline đã publish generation mới hơn (bình thường khi đang chạy).')
ctx.ui.record_step(ctx.work, 'lineage', 'PASS', run_id=start['run_id'])

In [ ]:
ctx = _ctx()  # [16/18] Kịch bản kiểm tra tương tác (tùy chọn, mặc định NOT_RUN): duplicate, late, tiến đồng hồ
# (stale), restart processor cùng checkpoint. Mỗi nút ghi thời điểm inject, kỳ vọng, giá trị trước/sau từ API và log.
start = ctx.require_start()
lab = ctx.ui.FaultLab(ctx.rt, ctx.api_base, start).show()

In [ ]:
ctx = _ctx()  # [17/18] Export artifacts: ZIP + manifest (commit, hashes, run/mode/scope, completeness, test statuses).
# Export khi session còn chạy là point-in-time (manifest ghi run_state=RUNNING). Không chứa token hay env dump.
# Export/tải xuống/sao lưu Drive lỗi: ghi FAIL hoặc cảnh báo, Run All vẫn tới cell 18. Cuối cell: tóm tắt lỗi màu đỏ.
if not ctx.p.export_artifacts:
    ctx.ui.record_step(ctx.work, 'export', 'NOT_RUN', reason='EXPORT_ARTIFACTS=False')
    print('NOT_RUN: EXPORT_ARTIFACTS=False')
else:
    zip_path = None
    export_args = ['--session-id', ctx.start.get('session_id')] if ctx.start and ctx.start.get('session_id') else []
    try:
        exported = ctx.run_cli('export_artifacts.py', *export_args, timeout=900, label='export_artifacts',
                               show=lambda d: ctx.ui.show_cli_result(d, 'Export artifacts', ('completeness', 'run_state')))
        zip_path = ctx.ui.export_zip_path(exported, ctx.work)
        zip_sha = ctx.ui.sha256_file(zip_path)
        reported_sha = ctx.ui.export_reported_sha(exported)
        if reported_sha and reported_sha != zip_sha:
            raise ctx.ui.CellFailed('EXPORT_CHECKSUM_MISMATCH', 'sha256 của ZIP khác giá trị export_artifacts báo cáo.')
    except ctx.ui.CellFailed as exc:
        zip_path = None
        ctx.ui.record_failure(ctx.work, 'export', exc)
        print('Export: FAIL (xem khung đỏ phía trên); bỏ qua tải xuống và sao lưu Drive.')
    if zip_path is not None:
        print('ZIP   :', zip_path)
        print('sha256:', zip_sha, '| size: %.1f MB' % (zip_path.stat().st_size / 1e6))
        ctx.ui.record_step(ctx.work, 'export', 'PASS', zip=str(zip_path), sha256=zip_sha)
        try:
            from google.colab import files as colab_files
        except ImportError:
            colab_files = None
            print('Không chạy trên Colab: tải ZIP thủ công từ đường dẫn trên.')
        if colab_files is not None:
            try:
                colab_files.download(str(zip_path))
            except Exception as exc:
                print('CẢNH BÁO: không tự tải được ZIP (%s: %s). Tải thủ công từ đường dẫn trên (Files panel).'
                      % (type(exc).__name__, exc))
        if not ctx.p.mount_drive:
            ctx.ui.record_step(ctx.work, 'drive_backup', 'NOT_RUN', reason='MOUNT_DRIVE=False')
        else:
            try:
                from google.colab import drive
                drive.mount('/content/drive')
                dest = ctx.ui.copy_finished_zip(zip_path, '/content/drive/MyDrive/TransitPulse_artifacts', zip_sha)
            except ImportError:
                ctx.ui.record_step(ctx.work, 'drive_backup', 'NOT_RUN', reason='không chạy trên Colab')
                print('MOUNT_DRIVE=True nhưng không chạy trên Colab: bỏ qua sao lưu Drive.')
            except Exception as exc:
                ctx.ui.record_failure(ctx.work, 'drive_backup', exc)
            else:
                ctx.ui.record_step(ctx.work, 'drive_backup', 'PASS', dest=str(dest))
                print('Đã sao lưu ZIP đã hoàn tất lên Drive:', dest, '(chỉ ZIP; workdir vẫn ở disk local)')
ctx.ui.show_failure_summary(ctx.work, 'sau export')

In [ ]:
ctx = _ctx()  # [18/18] Dừng (tùy chọn). Mặc định STOP_AFTER_RUN_ALL=False: UI vẫn tương tác, dùng các nút khi xong.
# 'Dừng session' (demo.py stop) giữ API, dashboard và tunnel (link công khai vẫn mở); 'Tắt tunnel' đóng mọi truy
# cập công khai (demo.py expose stop --target both); 'Dừng tất cả' (demo.py stop --all) dừng mọi thứ kể cả Kafka,
# Cassandra, API, dashboard và tunnel. Chỉ dừng processes do supervisor sở hữu; giữ data/checkpoint/inputs.
# STRICT_RUN_ALL=True: cuối cell báo lỗi nếu còn bước FAIL trong notebook_steps.json.
if ctx.p.stop_after_run_all:
    ctx.ui.close_active()
    stopped = ctx.run_cli('demo.py', 'stop', '--all', timeout=900, label='demo stop --all',
                          show=lambda d: ctx.ui.show_cli_result(d, 'Đã dừng', ('data_preserved',)))
    ctx.ui.record_step(ctx.work, 'stop', 'PASS', all=True, data_preserved=stopped.get('data_preserved'))
    ctx.ui.record_step(ctx.work, 'tunnel_stop', 'PASS', via='demo.py stop --all')
else:
    stop_panel = ctx.ui.StopPanel(ctx.rt)
    stop_panel.show()
    print('Session tiếp tục chạy tới hạn DEMO_DURATION_SECONDS; supervisor ghi trạng thái khi hết budget.')
    if ctx.p.enable_cloudflare_tunnel:
        print('Link công khai (nếu đã mở) vẫn hoạt động cho tới khi bấm Tắt tunnel hoặc Dừng tất cả.')
failed = ctx.ui.show_failure_summary(ctx.work, 'kết thúc Run All')
if ctx.p.strict_run_all and failed:
    raise ctx.ui.CellFailed('STRICT_RUN_ALL_FAILED', '%d bước đã FAIL: %s' % (len(failed), ', '.join(str(f.get('step')) for f in failed)),
                            hint='Xem các khung đỏ phía trên và reports/notebook/notebook_steps.json; sửa lỗi rồi chạy lại.')